# AI 임원보고 분석 (5~9번 전용)

- 이 노트북은 **요청항목 5~9번만** 분석합니다.
- 1~4번(로그인/신규가입/일자추이/배너전환)은 마케팅팀 분석 범위로 분리합니다.
- 결과표는 `print` 시 **마크다운 표 형식**으로 출력되어 바로 복붙 가능합니다.
- 시각화는 꼭 필요한 최소 그래프만 사용합니다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

def safe_div(n, d):
    if d is None or d == 0 or pd.isna(d):
        return np.nan
    return n / d

def print_md_table(title, df, digits=4):
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)
    if df is None or len(df) == 0:
        print("(데이터 없음)")
        return
    out = df.copy()
    num_cols = out.select_dtypes(include=["number"]).columns
    for c in num_cols:
        out[c] = out[c].round(digits)
    try:
        print(out.to_markdown(index=False))
    except Exception:
        print(out.to_string(index=False))

In [ ]:
# ===== 사용자 환경에 맞게 경로/오픈일 수정 =====
OPEN_DATE = pd.Timestamp("2026-03-23")
DATA_DIR = Path("data")

profile_file = DATA_DIR / "customer_profile.csv"
chat_file = DATA_DIR / "ai_chat_daily_by_user.csv"
ai_transfer_file = DATA_DIR / "ai_transfer_daily_by_user.csv"

In [ ]:
def load_csv(path):
    if path.exists():
        return pd.read_csv(path)
    print(f"[WARN] 파일 없음: {path}")
    return pd.DataFrame()

def first_existing_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def normalize_profile(df):
    if df.empty:
        return df
    mapper = {
        "customer_id": ["customer_id", "고객번호"],
        "age_band": ["age_band", "나이대"],
        "is_employee": ["is_employee", "임직원여부"],
        "ebank_signup_date": ["ebank_signup_date", "전자금융가입일", "first_account_open_date"],
        "ai_signup_date": ["ai_signup_date", "AI가입일", "ai_join_date"],
        "last_login_date": ["last_login_date", "최근접속일"],
        "pre_year_avg_transfer_count": ["pre_year_avg_transfer_count", "AI가입전1년_월평균이체건수"],
        "pre30_transfer_count": ["pre30_transfer_count", "AI가입전30일_이체건수"],
        "post30_ai_transfer_count": ["post30_ai_transfer_count", "AI가입후30일_ai이체건수"],
        "post30_other_transfer_count": ["post30_other_transfer_count", "AI가입후30일_기존이체건수"],
        "feedback_like_count": ["feedback_like_count", "피드백좋아요건"],
        "feedback_dislike_count": ["feedback_dislike_count", "피드백싫어요건"],
        "unanswered_count": ["unanswered_count", "답변불가경험건수"]
    }
    rename = {}
    for std, cands in mapper.items():
        found = first_existing_column(df, cands)
        if found:
            rename[found] = std
    out = df.rename(columns=rename).copy()

    for c in ["ebank_signup_date", "ai_signup_date", "last_login_date"]:
        if c in out.columns:
            out[c] = pd.to_datetime(out[c], errors="coerce")

    return out

def normalize_chat(df):
    if df.empty:
        return df
    mapper = {
        "customer_id": ["customer_id", "고객번호"],
        "ai_signup_date": ["ai_signup_date", "AI가입일"],
        "chat_date": ["chat_date", "채팅요청일", "요청일", "date"],
        "service_category": ["service_category", "서비스분류", "서비스분류코드"],
        "request_count": ["request_count", "건수", "cnt"]
    }
    rename = {}
    for std, cands in mapper.items():
        found = first_existing_column(df, cands)
        if found:
            rename[found] = std
    out = df.rename(columns=rename).copy()

    for c in ["ai_signup_date", "chat_date"]:
        if c in out.columns:
            out[c] = pd.to_datetime(out[c], errors="coerce")

    if "request_count" not in out.columns:
        out["request_count"] = 1
    out["request_count"] = pd.to_numeric(out["request_count"], errors="coerce").fillna(0)

    return out

def normalize_ai_transfer(df):
    if df.empty:
        return df
    mapper = {
        "customer_id": ["customer_id", "고객번호"],
        "ai_signup_date": ["ai_signup_date", "AI가입일"],
        "ai_transfer_date": ["ai_transfer_date", "ai이체일"],
        "ai_transfer_count": ["ai_transfer_count", "ai이체건수"],
        "ai_transfer_amount": ["ai_transfer_amount", "금액합"]
    }
    rename = {}
    for std, cands in mapper.items():
        found = first_existing_column(df, cands)
        if found:
            rename[found] = std
    out = df.rename(columns=rename).copy()

    for c in ["ai_signup_date", "ai_transfer_date"]:
        if c in out.columns:
            out[c] = pd.to_datetime(out[c], errors="coerce")

    if "ai_transfer_count" in out.columns:
        out["ai_transfer_count"] = pd.to_numeric(out["ai_transfer_count"], errors="coerce").fillna(0)
    else:
        out["ai_transfer_count"] = 0

    if "ai_transfer_amount" in out.columns:
        out["ai_transfer_amount"] = pd.to_numeric(out["ai_transfer_amount"], errors="coerce").fillna(0)
    else:
        out["ai_transfer_amount"] = 0

    return out

In [ ]:
profile = normalize_profile(load_csv(profile_file))
chat = normalize_chat(load_csv(chat_file))
ai_transfer = normalize_ai_transfer(load_csv(ai_transfer_file))

print_md_table("데이터 로딩 현황", pd.DataFrame([
    {"dataset": "profile", "rows": len(profile)},
    {"dataset": "chat", "rows": len(chat)},
    {"dataset": "ai_transfer", "rows": len(ai_transfer)},
]))

## 5) AI가입일 = 신규가입일(전자금융가입일) 일치 여부

In [ ]:
result_5 = pd.DataFrame()

if {"ai_signup_date", "ebank_signup_date"}.issubset(profile.columns):
    tmp = profile.dropna(subset=["ai_signup_date", "ebank_signup_date"]).copy()
    tmp["diff_days"] = (tmp["ai_signup_date"] - tmp["ebank_signup_date"]).dt.days

    def bucket(x):
        if pd.isna(x):
            return "unknown"
        if x == 0:
            return "동일일"
        if 1 <= x <= 7:
            return "가입후 7일 이내"
        if 8 <= x <= 30:
            return "가입후 8~30일"
        if x > 30:
            return "가입후 30일 초과"
        return "AI가입이 더 빠름(데이터확인)"

    tmp["구분"] = tmp["diff_days"].apply(bucket)
    result_5 = tmp.groupby("구분", as_index=False).agg(고객수=("customer_id", "nunique"))
    total = result_5["고객수"].sum()
    result_5["비중"] = result_5["고객수"].apply(lambda x: safe_div(x, total))

print_md_table("5) AI가입일 vs 신규가입일 분포", result_5)

## 6) AI가입자 대상 가입 전후 이체 사용 양상 비교 (실행 퍼널 포함)

In [ ]:
result_6a = pd.DataFrame()
result_6b = pd.DataFrame()

needed = {"pre30_transfer_count", "post30_ai_transfer_count", "post30_other_transfer_count"}
if needed.issubset(profile.columns):
    tmp = profile.copy()
    for c in list(needed):
        tmp[c] = pd.to_numeric(tmp[c], errors="coerce").fillna(0)

    tmp["post30_total_transfer_count"] = tmp["post30_ai_transfer_count"] + tmp["post30_other_transfer_count"]
    tmp["delta_total_transfer_count"] = tmp["post30_total_transfer_count"] - tmp["pre30_transfer_count"]

    result_6a = pd.DataFrame([
        {"지표": "가입 전 30일 평균 이체건수", "값": tmp["pre30_transfer_count"].mean()},
        {"지표": "가입 후 30일 평균 전체이체건수", "값": tmp["post30_total_transfer_count"].mean()},
        {"지표": "가입 후 30일 평균 AI이체건수", "값": tmp["post30_ai_transfer_count"].mean()},
        {"지표": "가입 후 30일 평균 기존이체건수", "값": tmp["post30_other_transfer_count"].mean()},
        {"지표": "가입 전후 평균 증감(건수)", "값": tmp["delta_total_transfer_count"].mean()},
    ])

ai_signup_users = profile["customer_id"].nunique() if "customer_id" in profile.columns else np.nan
request_users = chat.loc[chat["request_count"] > 0, "customer_id"].nunique() if not chat.empty else 0
if not chat.empty and "service_category" in chat.columns:
    req_transfer_users = chat[
        (chat["request_count"] > 0)
        & (chat["service_category"].astype(str).str.contains("이체|transfer", case=False, na=False))
    ]["customer_id"].nunique()
else:
    req_transfer_users = np.nan
exec_transfer_users = ai_transfer.loc[ai_transfer["ai_transfer_count"] > 0, "customer_id"].nunique() if not ai_transfer.empty else 0

result_6b = pd.DataFrame([
    {"퍼널단계": "AI가입자", "고객수": ai_signup_users, "전단계 전환율": np.nan},
    {"퍼널단계": "AI요청 경험자", "고객수": request_users, "전단계 전환율": safe_div(request_users, ai_signup_users)},
    {"퍼널단계": "AI이체 요청 경험자", "고객수": req_transfer_users, "전단계 전환율": safe_div(req_transfer_users, request_users)},
    {"퍼널단계": "AI이체 실행자", "고객수": exec_transfer_users, "전단계 전환율": safe_div(exec_transfer_users, req_transfer_users)},
])

print_md_table("6-A) 가입 전후 이체 사용 비교", result_6a)
print_md_table("6-B) AI이체 전환 퍼널", result_6b)

if not result_6b.empty:
    plt.figure(figsize=(8, 3.5))
    sns.barplot(data=result_6b, x="퍼널단계", y="고객수", color="#4C72B0")
    plt.title("AI이체 퍼널(가입→요청→실행)")
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()

## 7) AI가입자 재사용 현황 (의도 4개 합산 발화건수 기준)

In [ ]:
result_7 = pd.DataFrame()
reuse_curve = pd.DataFrame()

if not chat.empty and {"customer_id", "ai_signup_date", "chat_date", "request_count"}.issubset(chat.columns):
    c = chat.dropna(subset=["customer_id", "ai_signup_date", "chat_date"]).copy()
    c["day_offset"] = (c["chat_date"] - c["ai_signup_date"]).dt.days
    c = c[c["day_offset"] >= 0]

    user_day = c.groupby(["customer_id", "day_offset"], as_index=False).agg(total_requests=("request_count", "sum"))

    ai_signup_users = profile["customer_id"].nunique() if "customer_id" in profile.columns else c["customer_id"].nunique()
    used_once = user_day["customer_id"].nunique()
    reused_any = user_day[user_day["day_offset"] >= 1]["customer_id"].nunique()
    reused_7d = user_day[(user_day["day_offset"] >= 1) & (user_day["day_offset"] <= 7)]["customer_id"].nunique()
    reused_30d = user_day[(user_day["day_offset"] >= 1) & (user_day["day_offset"] <= 30)]["customer_id"].nunique()

    req_30 = user_day[user_day["day_offset"] <= 30].groupby("customer_id", as_index=False).agg(
        req_30d=("total_requests", "sum"),
        active_days_30d=("day_offset", "nunique")
    )
    users_2plus_req_30 = req_30[req_30["req_30d"] >= 2]["customer_id"].nunique()
    users_3plus_days_30 = req_30[req_30["active_days_30d"] >= 3]["customer_id"].nunique()

    result_7 = pd.DataFrame([
        {"지표": "AI가입자수", "값": ai_signup_users},
        {"지표": "가입 후 1회 이상 사용", "값": used_once, "비율": safe_div(used_once, ai_signup_users)},
        {"지표": "가입 후 다른날 재사용", "값": reused_any, "비율": safe_div(reused_any, ai_signup_users)},
        {"지표": "가입 후 7일 내 재사용", "값": reused_7d, "비율": safe_div(reused_7d, ai_signup_users)},
        {"지표": "가입 후 30일 내 재사용", "값": reused_30d, "비율": safe_div(reused_30d, ai_signup_users)},
        {"지표": "가입 후 30일 내 발화 2회 이상", "값": users_2plus_req_30, "비율": safe_div(users_2plus_req_30, ai_signup_users)},
        {"지표": "가입 후 30일 내 활동일 3일 이상", "값": users_3plus_days_30, "비율": safe_div(users_3plus_days_30, ai_signup_users)},
    ])

    curve_rows = []
    for d in range(1, 31):
        users_d = user_day[(user_day["day_offset"] >= 1) & (user_day["day_offset"] <= d)]["customer_id"].nunique()
        curve_rows.append({"day": d, "cum_reuse_rate": safe_div(users_d, ai_signup_users)})
    reuse_curve = pd.DataFrame(curve_rows)

print_md_table("7) AI 재사용 핵심지표", result_7)

if not reuse_curve.empty:
    plt.figure(figsize=(7.5, 3.5))
    sns.lineplot(data=reuse_curve, x="day", y="cum_reuse_rate", marker="o", color="#55A868")
    plt.title("가입 후 누적 재사용률 (1~30일)")
    plt.ylabel("누적 재사용률")
    plt.xlabel("가입 후 경과일")
    plt.tight_layout()
    plt.show()

## 8) AI가입 후 미사용자 현황

In [ ]:
result_8 = pd.DataFrame()
user_segment = pd.DataFrame()

if "customer_id" in profile.columns:
    base_users = profile[["customer_id"]].drop_duplicates().copy()

    if not chat.empty and {"customer_id", "ai_signup_date", "chat_date", "request_count"}.issubset(chat.columns):
        c = chat.dropna(subset=["customer_id", "ai_signup_date", "chat_date"]).copy()
        c["day_offset"] = (c["chat_date"] - c["ai_signup_date"]).dt.days
        c = c[c["day_offset"] >= 0]
        usage = c.groupby("customer_id", as_index=False).agg(
            total_requests=("request_count", "sum"),
            active_days=("chat_date", "nunique"),
            recent_chat_date=("chat_date", "max")
        )
    else:
        usage = pd.DataFrame(columns=["customer_id", "total_requests", "active_days", "recent_chat_date"])

    user_segment = base_users.merge(usage, on="customer_id", how="left")
    user_segment["total_requests"] = user_segment["total_requests"].fillna(0)
    user_segment["active_days"] = user_segment["active_days"].fillna(0)

    def segment_row(r):
        if r["total_requests"] == 0:
            return "미사용"
        if r["active_days"] == 1:
            return "1회성사용"
        return "재사용"

    user_segment["segment"] = user_segment.apply(segment_row, axis=1)
    result_8 = user_segment.groupby("segment", as_index=False).agg(고객수=("customer_id", "nunique"))
    total = result_8["고객수"].sum()
    result_8["비중"] = result_8["고객수"].apply(lambda x: safe_div(x, total))

print_md_table("8) AI가입 후 사용자 분류(미사용/1회성/재사용)", result_8)

## 9) 미사용자 vs 재사용자 특성 비교

In [ ]:
result_9_num = pd.DataFrame()
result_9_cat = pd.DataFrame()

if not user_segment.empty and not profile.empty and "customer_id" in profile.columns:
    comp = profile.merge(user_segment[["customer_id", "segment"]], on="customer_id", how="left")
    comp["segment2"] = np.where(comp["segment"].isin(["미사용", "1회성사용"]), "미사용/1회성", "재사용")

    num_candidates = [
        "pre_year_avg_transfer_count",
        "pre30_transfer_count",
        "feedback_like_count",
        "feedback_dislike_count",
        "unanswered_count",
    ]
    use_num = [c for c in num_candidates if c in comp.columns]

    rows = []
    for c in use_num:
        comp[c] = pd.to_numeric(comp[c], errors="coerce")
        g = comp.groupby("segment2")[c].mean()
        rows.append({
            "항목": c,
            "미사용/1회성 평균": g.get("미사용/1회성", np.nan),
            "재사용 평균": g.get("재사용", np.nan),
            "차이(재사용-미사용/1회성)": g.get("재사용", np.nan) - g.get("미사용/1회성", np.nan),
        })
    result_9_num = pd.DataFrame(rows)

    cat_frames = []
    if "age_band" in comp.columns:
        a = comp.groupby(["segment2", "age_band"], as_index=False).agg(고객수=("customer_id", "nunique"))
        a["구분"] = "age_band"
        a = a.rename(columns={"age_band": "카테고리값"})
        cat_frames.append(a[["구분", "segment2", "카테고리값", "고객수"]])

    if "is_employee" in comp.columns:
        b = comp.groupby(["segment2", "is_employee"], as_index=False).agg(고객수=("customer_id", "nunique"))
        b["구분"] = "is_employee"
        b = b.rename(columns={"is_employee": "카테고리값"})
        cat_frames.append(b[["구분", "segment2", "카테고리값", "고객수"]])

    if cat_frames:
        result_9_cat = pd.concat(cat_frames, ignore_index=True)

print_md_table("9-A) 미사용/1회성 vs 재사용 수치 특성 비교", result_9_num)
print_md_table("9-B) 미사용/1회성 vs 재사용 범주 특성 분포", result_9_cat)

## 보고용 한 줄 코멘트 자동 생성 (5~9번 전용)

In [ ]:
comments = []

if not result_5.empty and "구분" in result_5.columns:
    same = result_5.loc[result_5["구분"] == "동일일", "고객수"]
    total = result_5["고객수"].sum()
    same_ratio = safe_div(float(same.iloc[0]) if len(same) else 0, total)
    if pd.notna(same_ratio) and same_ratio >= 0.2:
        comments.append(f"- (5) 동일일 가입 비중이 {same_ratio:.1%}로, 신규고객 온보딩 구간에서 AI 진입 신호가 확인됩니다.")
    else:
        comments.append(f"- (5) 동일일 가입 비중은 {same_ratio:.1%} 수준으로 제한적이며, 다수는 앱 사용 경험 축적 후 AI에 진입하는 패턴입니다.")

if not result_7.empty:
    r30 = result_7.loc[result_7["지표"] == "가입 후 30일 내 재사용", "비율"]
    r30 = float(r30.iloc[0]) if len(r30) else np.nan
    if pd.notna(r30) and r30 >= 0.2:
        comments.append(f"- (7) 30일 재사용률이 {r30:.1%}로, 초기 체험을 넘어 반복 사용군이 형성되고 있습니다.")
    else:
        comments.append(f"- (7) 30일 재사용률은 {r30:.1%}로, 현재 핵심 과제는 가입 확대보다 첫 사용 이후 재방문 유도입니다.")

if not result_8.empty:
    no_use = result_8.loc[result_8["segment"] == "미사용", "비중"]
    no_use = float(no_use.iloc[0]) if len(no_use) else np.nan
    if pd.notna(no_use):
        comments.append(f"- (8) 미사용 비중은 {no_use:.1%}이며, 가입 직후 첫 성공경험 설계가 전환 개선의 우선 과제입니다.")

if not result_9_num.empty and "차이(재사용-미사용/1회성)" in result_9_num.columns:
    top = result_9_num.sort_values("차이(재사용-미사용/1회성)", ascending=False).head(1)
    if len(top):
        row = top.iloc[0]
        comments.append(f"- (9) 재사용군은 `{row['항목']}` 항목에서 상대적으로 높아, 타깃 세분화 기준으로 활용 가능합니다.")

print("
".join(comments) if comments else "코멘트 생성을 위한 결과 데이터가 부족합니다.")